In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv("./data/log_sensor.csv")
print(f"Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data: 1777 baris, 9 kolom


,Timestamp,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
0,2026-07-04T01:32:00Z,82.5,25.3,24.8,58.0,91,190,102,1750
1,2026-07-04T01:37:00Z,80.5,25.2,24.8,57.1,93,187,101,1750
2,2026-07-04T01:42:00Z,77.9,25.6,25.3,53.5,91,190,101,1750
3,2026-07-04T01:47:00Z,78.2,25.8,25.0,55.4,94,188,103,1750
4,2026-07-04T01:52:00Z,75.1,26.0,26.0,51.5,93,189,100,1758


# Cek data

In [2]:
df[["soil_moisture", "soil_temperature", "air_temperature",
    "air_humidity", "nitrogen", "fosfor", "kalium", "ec"]].describe().round(2)

,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
count,1777.00,1777.00,1777.00,1777.00,1777.00,1777.00,1777.00,1777.00
mean,63.24,23.98,24.64,61.33,87.53,170.10,92.01,2080.42
std,10.04,2.30,6.98,20.63,6.73,15.47,8.43,234.58
min,0.00,19.50,15.00,40.00,0.00,0.00,0.00,0.00
25%,57.70,22.00,17.50,40.00,86.00,164.00,88.00,1931.00
50%,60.80,24.00,24.80,57.00,88.00,170.00,93.00,2100.00
75%,65.30,25.90,31.70,83.10,90.00,179.00,97.00,2239.00
max,95.00,28.30,34.00,99.00,95.00,190.00,104.00,2393.00


# Normalisasi Skala Sensor

In [3]:
# ec: sensor output µS/cm, konversi ke mS/cm (bagi 100)

df["ec"] = df["ec"] / 100.0       

# df["kalium"] = df["kalium"] / 3.5

print("Setelah normalisasi:")
print(df[["nitrogen", "fosfor", "kalium", "ec"]].describe().round(2))

Setelah normalisasi:
       nitrogen   fosfor   kalium       ec
count   1777.00  1777.00  1777.00  1777.00
mean      87.53   170.10    92.01    20.80
std        6.73    15.47     8.43     2.35
min        0.00     0.00     0.00     0.00
25%       86.00   164.00    88.00    19.31
50%       88.00   170.00    93.00    21.00
75%       90.00   179.00    97.00    22.39
max       95.00   190.00   104.00    23.93


# Tambah Kolom Fase (butuh plant_age)

In [4]:
TANGGAL_TANAM = pd.Timestamp("2026-05-01", tz="UTC")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["plant_age"] = (df["Timestamp"] - TANGGAL_TANAM).dt.days

def get_fase(age):
    if age <= 30:   return 0    # establishment
    elif age <= 55: return 1    # vegetatif
    elif age <= 75: return 2    # berbunga (initial flowering - fruit set)
    else:           return 3    # pematangan (fruit development & maturation)

df["fase"] = df["plant_age"].apply(get_fase)

FASE_NAMES = {0: "establishment", 1: "vegetatif", 2: "berbunga", 3: "pematangan"}

print(df[["Timestamp", "plant_age", "fase"]].head())
print("\nDistribusi fase:")
for code, name in FASE_NAMES.items():
    n = (df["fase"] == code).sum()
    print(f"  {code} = {name:15s} {n}")

                  Timestamp  plant_age  fase
0 2026-07-04 01:32:00+00:00         64     2
1 2026-07-04 01:37:00+00:00         64     2
2 2026-07-04 01:42:00+00:00         64     2
3 2026-07-04 01:47:00+00:00         64     2
4 2026-07-04 01:52:00+00:00         64     2

Distribusi fase:
  0 = establishment   0
  1 = vegetatif       0
  2 = berbunga        1777
  3 = pematangan      0


# Rasio Ideal NPK per Fase (dari Haifa)

In [5]:
# Rasio N:P2O5:K2O ideal per fase (Haifa Crop Guide)
RASIO_IDEAL = {
    0: (1, 2, 1),   # establishment - P dominan (root development)
    1: (1, 1, 1),   # vegetatif - seimbang
    2: (2, 1, 3),   # berbunga - K dominan, P turun
    3: (2, 1, 3),   # pematangan - K dominan
}

# Batas absolut per unsur (mg/kg) - sesuaikan skala sensormu
BATAS_MIN = {"n": 40, "p": 50, "k": 60}
BATAS_MAX = {"n": 150, "p": 150, "k": 200}

print("Rasio ideal & batas siap.")

Rasio ideal & batas siap.


# Labelling dataset pupuk

In [6]:
LABEL_NAMES = {
    0: "Tidak perlu",
    1: "Urea/ZA",              # N kurang
    2: "SP-36",                # P kurang
    3: "KCl",                  # K kurang
    4: "Urea/ZA + SP-36",      # N,P kurang
    5: "Urea/ZA + KCl",        # N,K kurang
    6: "SP-36 + KCl",          # P,K kurang
    7: "Urea/ZA + SP-36 + KCl",# semua kurang
    8: "NPK 15-15-15",         # maintenance
    9: "Kurangi pemupukan N",  # N berlebih
    10: "Flush air (EC/nutrisi tinggi)",  # over/salinitas
}

def label_pupuk(row):
    n, p, k = row["nitrogen"], row["fosfor"], row["kalium"]
    ec = row["ec"]
    fase = row["fase"]   # sekarang int (0-3)

    # --- Cek kelebihan dulu ---
    if ec > 4.0:
        return 10
    if n > BATAS_MAX["n"] and k > BATAS_MAX["k"]:
        return 10
    if n > BATAS_MAX["n"]:
        return 9

    # --- Defisiensi berdasarkan rasio fase ---
    rn, rp, rk = RASIO_IDEAL[fase]   # akses pakai int
    total_ratio = rn + rp + rk

    total_npk = n + p + k
    if total_npk == 0:
        return 7

    prop_n, prop_p, prop_k = n/total_npk, p/total_npk, k/total_npk
    ideal_n, ideal_p, ideal_k = rn/total_ratio, rp/total_ratio, rk/total_ratio

    TOLERANSI = 0.7
    n_low = (prop_n < ideal_n * TOLERANSI) or (n < BATAS_MIN["n"])
    p_low = (prop_p < ideal_p * TOLERANSI) or (p < BATAS_MIN["p"])
    k_low = (prop_k < ideal_k * TOLERANSI) or (k < BATAS_MIN["k"])

    if   n_low and p_low and k_low: return 7
    elif n_low and p_low:           return 4
    elif n_low and k_low:           return 5
    elif p_low and k_low:           return 6
    elif n_low:                     return 1
    elif p_low:                     return 2
    elif k_low:                     return 3
    else:                           return 0

print("Fungsi label_pupuk() siap.")

Fungsi label_pupuk() siap.


In [7]:
df["recommendation"] = df.apply(label_pupuk, axis=1)
df["recommendation_label"] = df["recommendation"].map(LABEL_NAMES)

print("Distribusi rekomendasi:")
dist = df["recommendation"].value_counts().sort_index()
for code, count in dist.items():
    print(f"  {code} = {LABEL_NAMES[code]:35s} {count:5d} ({count/len(df)*100:.1f}%)")

Distribusi rekomendasi:
  7 = Urea/ZA + SP-36 + KCl                   9 (0.5%)
  10 = Flush air (EC/nutrisi tinggi)        1768 (99.5%)


# Preview Hasil

In [8]:
df["fase_label"] = df["fase"].map(FASE_NAMES)
df[["Timestamp", "nitrogen", "fosfor", "kalium", "ec",
    "fase_label", "recommendation_label"]].head(20)

,Timestamp,nitrogen,fosfor,kalium,ec,fase_label,recommendation_label
0,2026-07-04 01:32:00+00:00,91,190,102,17.50,berbunga,Flush air (EC/nutrisi tinggi)
1,2026-07-04 01:37:00+00:00,93,187,101,17.50,berbunga,Flush air (EC/nutrisi tinggi)
2,2026-07-04 01:42:00+00:00,91,190,101,17.50,berbunga,Flush air (EC/nutrisi tinggi)
3,2026-07-04 01:47:00+00:00,94,188,103,17.50,berbunga,Flush air (EC/nutrisi tinggi)
4,2026-07-04 01:52:00+00:00,93,189,100,17.58,berbunga,Flush air (EC/nutrisi tinggi)
5,2026-07-04 01:57:00+00:00,92,186,103,17.50,berbunga,Flush air (EC/nutrisi tinggi)
6,2026-07-04 02:02:00+00:00,94,186,102,17.57,berbunga,Flush air (EC/nutrisi tinggi)
7,2026-07-04 02:07:00+00:00,94,187,103,17.54,berbunga,Flush air (EC/nutrisi tinggi)
8,2026-07-04 02:12:00+00:00,93,189,103,17.60,berbunga,Flush air (EC/nutrisi tinggi)
9,2026-07-04 02:17:00+00:00,93,186,101,17.62,berbunga,Flush air (EC/nutrisi tinggi)


# Simpan dataset

In [9]:
sensor_cols = ["soil_moisture", "soil_temperature", "air_temperature",
               "air_humidity", "nitrogen", "fosfor", "kalium", "ec"]

# Dataset pupuk: 7 fitur + label
cols_pupuk = ["nitrogen", "fosfor", "kalium", "plant_age", "fase", "ec", "soil_moisture", "recommendation"]
df[cols_pupuk].to_csv("data/dataset_pupuk.csv", index=False)

print("Tersimpan: dataset_pupuk.csv")
print(f"Total: {len(df)} baris")

Tersimpan: dataset_pupuk.csv
Total: 1777 baris
